In [ ]:
import os
import json
import time
from cryptography.fernet import Fernet 
import base64 #generate nonce

# Generate encryption keys for each component
key_database = {
    "client": Fernet.generate_key(),
    "AS": Fernet.generate_key(),
    "TGS": Fernet.generate_key(),
    "service": Fernet.generate_key()
}

# ==========================
# 🔐 Encryption & Decryption
# ==========================

def encrypt_data(key, data):
    """Encrypts a JSON object using Fernet symmetric encryption."""
    # TODO: Implement encryption using Fernet
    encrypt_instance = Fernet(key)
    json_str = json.dumps(data)
    encoded_data = json_str.encode('utf-8')
    encrypted_data = encrypt_instance.encrypt(encoded_data)
    return encrypted_data

def decrypt_data(key, encrypted_data):
    """Decrypts a JSON object using Fernet symmetric encryption."""
    # TODO: Implement decryption using Fernet
    decrypt_instance = Fernet(key)
    decrypted_bytes = decrypt_instance.decrypt(encrypted_data)
    decrypted_str = decrypted_bytes.decode('utf-8')
    decrypted_data = json.loads(decrypted_str)
    return decrypted_data

# ==========================
# 🎟️ Authentication Server (AS) - Issues Ticket Granting Ticket (TGT)
# ==========================

def authentication_server(client_id):
    """Simulates the Authentication Server issuing a TGT to the client."""
    # TODO: Generate a session key and a nonce
    session_key = Fernet.generate_key()
    nonce = base64.urlsafe_b64encode(os.urandom(16))

    # TODO: Define what a TGT is
    #TGT -> ticket granting ticket. A ticket given to the client 
    #   after the Auth Server authenticates them so the client can
    #   talk to the TGS.
    #   Contains the client_id, session key, nonce, timestamp, and expiration time

    # TODO: Encrypt the TGT with the AS key before returning
    
    tgt_data = {
        "client_id": client_id,
        "session_key": session_key.decode('utf-8'), 
        "nonce": nonce.decode('utf-8'),
        "issue_time": time.time(),
        "expires_in": time.time() + 300   #5 minutes from current
    }  # Replace with encryption function
    AS_key = key_database["AS"] 
    encrypted_tgt = encrypt_data(AS_key, tgt_data)

    return encrypted_tgt, session_key

# ==========================
# 🎫 Ticket Granting Server (TGS) - Issues Service Ticket
# ==========================

def ticket_granting_server(encrypted_tgt, client_session_key):
    """Simulates the Ticket Granting Server issuing a service ticket."""
    try:
        # TODO: Decrypt the TGT using AS's key
        AS_key = key_database["AS"] 
        decrypted_tgt = decrypt_data(AS_key, encrypted_tgt)  # Replace with decryption function

        # TODO: Validate ticket expiration (reject if > 5 minutes old)
        current = time.time()
        if current > decrypted_tgt["expires_in"]:
            return Exception("Ticket is expired")

        # TODO: Generate a session key and a nonce
        service_session_key = encrypt_data(client_session_key, Fernet.generate_key().decode('utf-8')) 
        nonce = base64.urlsafe_b64encode(os.urandom(16))

        # TODO: Define a Service Ticket
        # a ticket issued by the TGS to the client once TGT is authenticated
        # gives client temporary access to the designated server
        # contains client id, session key, nonce, issued time, and expiration time
        # a service ID is usually included but for this case it is not generated
        service_data = {
            "client_id": decrypted_tgt["client_id"],
            "session_key": service_session_key.decode('utf-8'),
            "nonce": nonce.decode('utf-8'),
            "issue_time": current,
            "expires_in": current +300
        }

        # TODO: Encrypt service ticket with TGS key before returning
        tgs_key = key_database["TGS"]
        encrypted_service_ticket = encrypt_data(tgs_key, service_data)  # Replace with encryption function

        return encrypted_service_ticket, service_session_key

    except Exception as e:
        return f"❌ Error in TGS: {str(e)}", None

# ==========================
# 🏛️ Service Server - Grants Access
# ==========================

def service_server(encrypted_service_ticket):
    """Validates and decrypts the service ticket before granting access."""
    try:
        # TODO: Decrypt service ticket using TGS's key
        tgs_key = key_database["TGS"]
        decrypted_ticket = decrypt_data(tgs_key, encrypted_service_ticket)  # Replace with decryption function

        # TODO: Validate timestamp (Reject if expired)
        current = time.time()
        if current > decrypted_ticket["expires_in"]:
            return Exception("Ticket is expired")

        # TODO: Return info extracted from the ticket
        client_id = decrypted_ticket["client_id"]
        session_key = decrypted_ticket["session_key"]
        return f"✅ Access granted to client ID: [{client_id}] with key [{session_key}]"

    except Exception as e:
        return f"❌ Service Authentication Failed: {str(e)}"

# ==========================
# 🖥️ Interactive CLI
# ==========================

def main():
    """Interactive command-line interface for running Kerberos authentication steps."""
    client_id = "client"
    tgt = None
    client_session_key = None
    service_ticket = None
    service_session_key = None

    while True:
        print("\n===== Kerberos Authentication System =====")
        print("1. Request Ticket Granting Ticket (TGT)")
        print("2. Request Service Ticket")
        print("3. Access Protected Service")
        print("4. Exit")
        choice = input("Enter your choice: ").strip()

        if choice == "1":
            # TODO get the TGT and client session key, print both
            tgt, client_session_key = authentication_server(client_id)
            print(f"\n🔐 Received ... ")
            print(f"TGT: [{tgt}], client session key: [{client_session_key}]")

        elif choice == "2":
            # TODO get the service ticket and service session key, print both
            service_ticket, service_session_key = ticket_granting_server(tgt, client_session_key)
            print(f"\n🎟️ Received ... ")
            print(f"Service Ticket: [{service_ticket}], service session key: [{service_session_key}]")

        elif choice == "3":
            # TODO access the service, print the result
            message = service_server(service_ticket)
            print(f"\n Received ... ")
            print(message)

        elif choice == "4":
            print("\n🚪 Exiting the system.")
            break

        else:
            print("\n❌ Invalid choice, please enter a valid option.")

# Run the interactive simulation
if __name__ == "__main__":
    main()



===== Kerberos Authentication System =====
1. Request Ticket Granting Ticket (TGT)
2. Request Service Ticket
3. Access Protected Service
4. Exit



🔐 Received ... 
TGT: [b'gAAAAABpgmWfPtTr1CXUs9wh60xzQZP8n-ho9P_tAzTwyqvgvJ6DpYFUws7B94yA44kSd77Ltz_BhDmMR7Vg54zO5RkKQ7dElAHmHSznT4aKAx0V6em7Vvoa4p-iLe95wy2Z2tJAjNY0O-J1VCmwLKBpbBxvT7xVF-oSotgUy7vZeT2c3Y4j8ppHXvRJaaU_ypHDaePZy5j1tBXfxfixDqOMV4Z5Y1Dluj4MPzAmodtmqtsyARlEhwjE5VBLOcrW07F-LVI-n6XbNcxGGXRxoWt7M_gqU6K-XmnXXLTYv3pW0KfvzOvUrn-3IYwpvSrK7yZ3nNcau6Zj'], client session key: [b'QDluQF1dQwPCY2qtaml-J-7p-HeI-eNIUt7XORwmlDU=']

===== Kerberos Authentication System =====
1. Request Ticket Granting Ticket (TGT)
2. Request Service Ticket
3. Access Protected Service
4. Exit

🎟️ Received ... 
Service Ticket: [b'gAAAAABpgmWk3LPDMovq8ocd9LOBjCp7opPFWvv7LdnZw647-8PPthlCoxSLvDzqZMxuTYWoJO7f-qGKlizwr8ofeO9kCf7Khyu308gFbLsbYepomCdszk155JsuRAtEbLYVUaQj5FjTw7rJws0LftSBKB1IbmBI02VqOIhhF0Rq6BtJEkg6s4l-KrUKHxFCY9q83cRrmRxwgS8xflvDBaWe7mBKq1CiJ2mcIsf-Fi52WrZNJ-5jn1toZq0GCEeNjPOLmpV-_8Bw-fz5edieOXR1kVJw9JLUO8Up4HOSia4mP2WWV5m88ySHyiyEqwgsyac_kfHLEwdDcG2O7hG8gvZBFBrBWqYL6qsyw_7aneoeWFd-iwZBtuVXVjQnEixKYZ